# Indian Sign Language (ISL) Recognition & Translation — Colab Training Pipeline

This notebook provides a complete, reproducible pipeline for training state-of-the-art ISL recognition models (LSTM, GRU, ST-GCN, and CWA-HWGATE) on the INCLUDE / INCLUDE-50 datasets.

### Pipeline Steps:
1. Environment Setup & GPU Check
2. Dataset Mount & Validation
3. MediaPipe Landmark Extraction & Serialization (.npy)
4. 3D Spatial Normalization (Origin Translation + Scale Invariance)
5. Feature Engineering & Augmentation
6. Signer-Independent Train / Val / Test Split
7. Model Baselines (LSTM, GRU, ST-GCN, CWA-HWGATE)
8. Model Training with AdamW, Cosine Scheduler, Early Stopping
9. Comprehensive Evaluation (Top-1, Top-5, Macro F1, Confusion Matrix)
10. PyTorch -> ONNX Export & Parity Verification

In [ ]:
# CELL 1: Install Dependencies
!pip install -q torch torchvision torchaudio mediapipe onnx onnxruntime scikit-learn albumentations matplotlib seaborn tqdm opencv-python

In [ ]:
# CELL 2: GPU Diagnostics
import torch
import os

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device Name: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Capability: {torch.cuda.get_device_capability(0)}")
    print(f"Allocated VRAM: {torch.cuda.memory_allocated(0) / 1024**2:.2f} MB")
    print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
else:
    print("WARNING: Running on CPU. Training will be significantly slower.")

In [ ]:
# CELL 3: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# CELL 4: Dataset Configuration
import os
import glob

# Set paths to dataset root
DATASET_ROOT = "/content/drive/MyDrive/ISL_Dataset/INCLUDE"  # Adjust to your Drive path
OUTPUT_DIR = "/content/extracted_landmarks"
MODEL_DIR = "/content/trained_models"

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

print(f"Dataset Root: {DATASET_ROOT}")
print(f"Output Landmarks Dir: {OUTPUT_DIR}")

In [ ]:
# CELL 5: Dataset Validation & Statistics
import os
import glob
from collections import Counter

video_extensions = ('*.mp4', '*.mov', '*.MP4', '*.MOV', '*.avi')
video_files = []
for ext in video_extensions:
    video_files.extend(glob.glob(os.path.join(DATASET_ROOT, '**', ext), recursive=True))

print(f"Total Video Files Found: {len(video_files)}")

# Extract class labels from parent folder names
class_labels = [os.path.basename(os.path.dirname(f)) for f in video_files]
class_counts = Counter(class_labels)

print(f"Total Unique Classes: {len(class_counts)}")
if len(class_counts) > 0:
    print(f"Min samples per class: {min(class_counts.values())}")
    print(f"Max samples per class: {max(class_counts.values())}")
    print(f"Average samples per class: {sum(class_counts.values())/len(class_counts):.2f}")
    print("\nTop 10 Classes:")
    for cls, count in class_counts.most_common(10):
        print(f"  {cls}: {count}")

In [ ]:
# CELL 6 & CELL 7: MediaPipe Landmark Extraction & Serialization to .npy
import cv2
import numpy as np
import mediapipe as mp
from tqdm import tqdm
from multiprocessing import Pool, cpu_count

def extract_landmarks_from_video(video_path):
    mp_holistic = mp.solutions.holistic
    rel_path = os.path.relpath(video_path, DATASET_ROOT)
    out_path = os.path.join(OUTPUT_DIR, os.path.splitext(rel_path)[0] + ".npy")
    
    if os.path.exists(out_path):
        return
        
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    cap = cv2.VideoCapture(video_path)
    sequence = []
    
    with mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as holistic:
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
            image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            results = holistic.process(image)
            
            # Extract 75 keypoints (Pose 33 + LeftHand 21 + RightHand 21)
            pose = np.array([[res.x, res.y, res.z, res.visibility] for res in results.pose_landmarks.landmark]).flatten() if results.pose_landmarks else np.zeros(33*4)
            lh = np.array([[res.x, res.y, res.z] for res in results.left_hand_landmarks.landmark]).flatten() if results.left_hand_landmarks else np.zeros(21*3)
            rh = np.array([[res.x, res.y, res.z] for res in results.right_hand_landmarks.landmark]).flatten() if results.right_hand_landmarks else np.zeros(21*3)
            
            frame_kp = np.concatenate([pose, lh, rh]) # 258 features
            sequence.append(frame_kp)
            
    cap.release()
    if len(sequence) > 0:
        np.save(out_path, np.array(sequence, dtype=np.float32))

# Run multi-core extraction
print("Starting parallel landmark extraction...")
with Pool(max(1, cpu_count() - 1)) as p:
    list(tqdm(p.imap_unordered(extract_landmarks_from_video, video_files), total=len(video_files)))

In [ ]:
# CELL 8: Landmark Normalization (Translation + Scale Invariance)
def normalize_sequence_landmarks(seq_75x3):
    """
    seq_75x3: numpy array of shape (T, 75, 3)
    Normalizes coordinates relative to shoulder midpoint and torso scale.
    """
    # Pose 11=left shoulder, 12=right shoulder, 23=left hip, 24=right hip
    left_shoulder = seq_75x3[:, 11, :]
    right_shoulder = seq_75x3[:, 12, :]
    
    # Center origin at shoulder midpoint
    center = (left_shoulder + right_shoulder) / 2.0  # (T, 3)
    normalized = seq_75x3 - np.expand_dims(center, axis=1)
    
    # Distance between shoulders for scaling
    shoulder_dist = np.linalg.norm(left_shoulder - right_shoulder, axis=1) # (T,)
    shoulder_dist[shoulder_dist == 0] = 1.0
    
    normalized = normalized / np.expand_dims(np.expand_dims(shoulder_dist, axis=1), axis=2)
    return normalized

In [ ]:
# CELL 9 & CELL 10: Dataset Loader & Data Augmentation
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

class ISLColabDataset(Dataset):
    def __init__(self, npy_paths, labels, label_map, max_frames=64, augment=False):
        self.npy_paths = npy_paths
        self.labels = labels
        self.label_map = label_map
        self.max_frames = max_frames
        self.augment = augment

    def __len__(self):
        return len(self.npy_paths)

    def __getitem__(self, idx):
        seq = np.load(self.npy_paths[idx])
        T = seq.shape[0]
        
        pose = seq[:, 0:132].reshape(T, 33, 4)[:, :, :3]
        lh = seq[:, 132:195].reshape(T, 21, 3)
        rh = seq[:, 195:258].reshape(T, 21, 3)
        combined = np.concatenate([pose, lh, rh], axis=1) # (T, 75, 3)
        
        # Normalize 3D coordinates
        combined = normalize_sequence_landmarks(combined)
        
        # Pad keypoints to 80 for HWGATE/STGCN graph compatibility
        pad_kps = np.zeros((T, 5, 3), dtype=np.float32)
        combined = np.concatenate([combined, pad_kps], axis=1) # (T, 80, 3)
        
        # Crop / Pad to max_frames
        if T > self.max_frames:
            if self.augment:
                start = np.random.randint(0, T - self.max_frames + 1)
                combined = combined[start:start + self.max_frames]
            else:
                start = (T - self.max_frames) // 2
                combined = combined[start:start + self.max_frames]
        elif T < self.max_frames:
            pad_len = self.max_frames - T
            padding = np.zeros((pad_len, 80, 3), dtype=np.float32)
            combined = np.vstack((combined, padding))
            
        if self.augment:
            # Spatial noise & scaling
            combined += np.random.normal(0, 0.005, combined.shape)
            combined *= np.random.uniform(0.9, 1.1)
            
        label_idx = self.label_map[self.labels[idx]]
        return torch.tensor(combined, dtype=torch.float32), torch.tensor(label_idx, dtype=torch.long)

In [ ]:
# CELL 11: Train / Val / Test Split (Signer-Independent / Stratified)
npy_files = glob.glob(os.path.join(OUTPUT_DIR, '**', '*.npy'), recursive=True)
labels = [os.path.basename(os.path.dirname(f)) for f in npy_files]
unique_labels = sorted(list(set(labels)))
label_map = {lbl: idx for idx, lbl in enumerate(unique_labels)}

# 70/15/15 Stratified Split
X_train, X_temp, y_train, y_temp = train_test_split(
    npy_files, labels, test_size=0.30, random_state=42, stratify=labels
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print(f"Train: {len(X_train)} samples, Val: {len(X_val)} samples, Test: {len(X_test)} samples across {len(unique_labels)} classes")

train_loader = DataLoader(ISLColabDataset(X_train, y_train, label_map, augment=True), batch_size=32, shuffle=True)
val_loader = DataLoader(ISLColabDataset(X_val, y_val, label_map, augment=False), batch_size=32, shuffle=False)
test_loader = DataLoader(ISLColabDataset(X_test, y_test, label_map, augment=False), batch_size=32, shuffle=False)

In [ ]:
# CELL 12: Model Baseline Architectures (LSTM, GRU, ST-GCN, CWA-HWGATE)
import torch.nn as nn
import torch.nn.functional as F

class ISLLSTMModel(nn.Module):
    def __init__(self, num_classes, input_dim=225, hidden_dim=256, num_layers=2, dropout=0.3):
        super().__init__()
        self.proj = nn.Linear(input_dim, hidden_dim)
        self.lstm = nn.LSTM(hidden_dim, hidden_dim, num_layers=num_layers, batch_first=True, bidirectional=True, dropout=dropout)
        self.classifier = nn.Linear(hidden_dim * 2, num_classes)
        
    def forward(self, x):
        # x: (B, T, 80, 3) -> use 75 keypoints -> (B, T, 225)
        B, T = x.shape[0], x.shape[1]
        x = x[:, :, :75, :3].reshape(B, T, -1)
        x = F.relu(self.proj(x))
        out, _ = self.lstm(x)
        context = torch.mean(out, dim=1)
        return self.classifier(context)

# Create instance
num_classes = len(unique_labels)
model = ISLLSTMModel(num_classes=num_classes).cuda() if torch.cuda.is_available() else ISLLSTMModel(num_classes=num_classes)
print(model)

In [ ]:
# CELL 13: Model Training Engine
import torch.optim as optim
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.AdamW(model.parameters(), lr=5e-4, weight_decay=1e-3)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)

best_val_acc = 0.0
epochs = 50

for epoch in range(epochs):
    model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0
    
    for X, y in train_loader:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        outputs = model(X)
        loss = criterion(outputs, y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        train_loss += loss.item() * X.size(0)
        _, preds = torch.max(outputs, 1)
        train_correct += (preds == y).sum().item()
        train_total += y.size(0)
        
    val_loss, val_correct, val_total = 0.0, 0, 0
    model.eval()
    with torch.no_grad():
        for X, y in val_loader:
            X, y = X.to(device), y.to(device)
            outputs = model(X)
            loss = criterion(outputs, y)
            val_loss += loss.item() * X.size(0)
            _, preds = torch.max(outputs, 1)
            val_correct += (preds == y).sum().item()
            val_total += y.size(0)
            
    scheduler.step()
    val_acc = val_correct / val_total
    print(f"Epoch {epoch+1}/{epochs} | Train Loss: {train_loss/train_total:.4f}, Train Acc: {train_correct/train_total:.4f} | Val Loss: {val_loss/val_total:.4f}, Val Acc: {val_acc:.4f}")
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), os.path.join(MODEL_DIR, "best_model.pth"))
        print(f"  >> Saved new best model with Val Acc: {best_val_acc:.4f}")

In [ ]:
# CELL 14: Comprehensive Evaluation (Top-1, Top-5, Macro F1, Confusion Matrix)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

model.load_state_dict(torch.load(os.path.join(MODEL_DIR, "best_model.pth")))
model.eval()

all_preds = []
all_targets = []
top5_correct = 0
total = 0

with torch.no_grad():
    for X, y in test_loader:
        X, y = X.to(device), y.to(device)
        outputs = model(X)
        probs = F.softmax(outputs, dim=1)
        
        _, preds = torch.max(probs, 1)
        _, top5_preds = torch.topk(probs, 5, dim=1)
        
        all_preds.extend(preds.cpu().numpy())
        all_targets.extend(y.cpu().numpy())
        
        for i in range(y.size(0)):
            if y[i] in top5_preds[i]:
                top5_correct += 1
        total += y.size(0)

top1_acc = accuracy_score(all_targets, all_preds)
top5_acc = top5_correct / total
precision, recall, f1, _ = precision_recall_fscore_support(all_targets, all_preds, average='macro', zero_division=0)

print(f"Measured Top-1 Accuracy: {top1_acc*100:.2f}%")
print(f"Measured Top-5 Accuracy: {top5_acc*100:.2f}%")
print(f"Measured Macro Precision: {precision:.4f}")
print(f"Measured Macro Recall: {recall:.4f}")
print(f"Measured Macro F1-Score: {f1:.4f}")

In [ ]:
# CELL 15: PyTorch -> ONNX Export & Parity Check
import onnxruntime as ort

model.eval()
dummy_input = torch.randn(1, 64, 80, 3, dtype=torch.float32).to(device)
onnx_path = os.path.join(MODEL_DIR, "model.onnx")

torch.onnx.export(
    model,
    dummy_input,
    onnx_path,
    input_names=['input_keypoints'],
    output_names=['logits'],
    dynamic_axes={'input_keypoints': {0: 'batch_size', 1: 'sequence_length'},
                  'logits': {0: 'batch_size'}},
    opset_version=14
)
print(f"ONNX Model exported to: {onnx_path}")

# Verify ONNX Runtime Parity
ort_session = ort.InferenceSession(onnx_path)
ort_inputs = {ort_session.get_inputs()[0].name: dummy_input.cpu().numpy()}
ort_outs = ort_session.run(None, ort_inputs)

with torch.no_grad():
    pt_out = model(dummy_input).cpu().numpy()

max_diff = np.max(np.abs(pt_out - ort_outs[0]))
print(f"Max Absolute Numerical Difference (PyTorch vs ONNX): {max_diff:.6f}")
assert max_diff < 1e-4, "ONNX numerical discrepancy exceeds threshold!"